# Pre-Reqs:

In [15]:
#pip install catboost optuna optuna-integration

import pandas as pd
import numpy as np
import optuna

from optuna.integration import CatBoostPruningCallback
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

# Data:

In [16]:
TrainDataset = pd.read_csv('/content/train.csv')
TestDataset = pd.read_csv('/content/test.csv')

TrainDataset.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0


In [9]:
#Tested, no missing values

In [17]:
#Values needing to change to numerical in dataset
Features = ["gender", "course","internet_access", "sleep_quality", "study_method", "facility_rating", "exam_difficulty"]

label_encoders = {}
for col in Features:
    le = LabelEncoder()

    # Make sure all values are strings
    TrainDataset[col] = TrainDataset[col].astype(str)
    TestDataset[col] = TestDataset[col].astype(str)

    # Fit on all unique SMILES
    all_smiles = pd.concat([TrainDataset[col], TestDataset[col]], axis=0).unique()
    le.fit(all_smiles)

    # Transform
    TrainDataset[col] = le.transform(TrainDataset[col])
    TestDataset[col] = le.transform(TestDataset[col])

    label_encoders[col] = le

#Numerical check
Train_Numerical = TrainDataset.applymap(lambda x: isinstance(x, (int, float))).all().all()
print(f"Test Dataset Numerical? {Train_Numerical}")
Test_Numerical = TestDataset.applymap(lambda x: isinstance(x, (int, float))).all().all()
print(f"Test Dataset Numerical? {Test_Numerical}")

/tmp/ipython-input-441121123.py:23: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  Train_Numerical = TrainDataset.applymap(lambda x: isinstance(x, (int, float))).all().all()


Test Dataset Numerical? True


/tmp/ipython-input-441121123.py:25: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  Test_Numerical = TestDataset.applymap(lambda x: isinstance(x, (int, float))).all().all()


Test Dataset Numerical? True


# Model 1 - CatBoost:

In [ ]:
'''
######################################Model, K-fold, Optuna and Output#####################################
'''
Target = "exam_score"
IDCol = "id"

#Data

X = TrainDataset.drop(columns=["exam_score", "id"]).copy()
y = TrainDataset["exam_score"].copy()
X_test = TestDataset.drop(columns=["id"]).copy()

categorical_features = [
  "age", "gender",	"course",	"study_hours", "class_attendance"	, "internet_access"	, "sleep_hours" ,	"sleep_quality"	, "study_method",	"facility_rating"	, "exam_difficulty"
 ]

for col in categorical_features:
  X[col] = X[col].astype(str)
  X_test[col] = X_test[col].astype(str)

cat_features = [X.columns.get_loc(col) for col in categorical_features]

def objective(trial):

    params = {
        "loss_function": "RMSE",
        "iterations": trial.suggest_int("iterations", 300, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "random_seed": 42,
        "verbose": 100,
        "allow_writing_files": False
    }

    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))

    for train_idx, valid_idx in kf.split(X, y):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = CatBoostRegressor(**params)

        model.fit(
            X_train, y_train,
            eval_set=(X_valid, y_valid),
            cat_features=cat_features,
            early_stopping_rounds=50,
            callbacks=[CatBoostPruningCallback(trial, "RMSE")]
        )

        oof_preds[valid_idx] = model.predict(X_valid)

    return np.sqrt(np.mean((oof_preds - y) ** 2))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

print("Best RMSE:", study.best_value)
print("Best parameters:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

#Retrain on all rows
best_params = study.best_params.copy()
best_params.update({
    "loss_function": "RMSE",
    "random_seed": 42,
    "verbose": False
})

final_model = CatBoostRegressor(**best_params)

final_model.fit(
    X, y,
    cat_features=cat_features
)

#Preditcion and injection into DF
test_preds = final_model.predict(X_test)

submission = pd.DataFrame({
    "id": TestDataset["id"],
    "exam_score": test_preds
})

#To CSV
submission.to_csv("KagglePSTSCatBoost.csv", index=False)

[I 2026-01-27 15:28:44,332] A new study created in memory with name: no-name-8bd1b3bc-6997-468d-b2b4-abf8388dbdeb


  0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipython-input-621681124.py:50: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  callbacks=[CatBoostPruningCallback(trial, "RMSE")]


0:	learn: 18.5360289	test: 18.4785090	best: 18.4785090 (0)	total: 1.45s	remaining: 8m 56s
100:	learn: 9.6118190	test: 9.4075597	best: 9.4075597 (100)	total: 1m 28s	remaining: 3m 55s
200:	learn: 9.4817064	test: 9.2873715	best: 9.2873715 (200)	total: 3m 14s	remaining: 2m 44s
300:	learn: 9.4498672	test: 9.2628630	best: 9.2628630 (300)	total: 5m 18s	remaining: 1m 14s
370:	learn: 9.4375105	test: 9.2534548	best: 9.2534548 (370)	total: 6m 44s	remaining: 0us

bestTest = 9.253454829
bestIteration = 370

0:	learn: 18.5210871	test: 18.5073219	best: 18.5073219 (0)	total: 1.05s	remaining: 6m 30s


/usr/local/lib/python3.12/dist-packages/optuna_integration/catboost/catboost.py:103: UserWarning: The reported value is ignored because this `step` 0 is already reported.
  self._trial.report(current_score, step=step)
/usr/local/lib/python3.12/dist-packages/optuna_integration/catboost/catboost.py:103: UserWarning: The reported value is ignored because this `step` 1 is already reported.
  self._trial.report(current_score, step=step)
/usr/local/lib/python3.12/dist-packages/optuna_integration/catboost/catboost.py:103: UserWarning: The reported value is ignored because this `step` 2 is already reported.
  self._trial.report(current_score, step=step)
/usr/local/lib/python3.12/dist-packages/optuna_integration/catboost/catboost.py:103: UserWarning: The reported value is ignored because this `step` 3 is already reported.
  self._trial.report(current_score, step=step)
/usr/local/lib/python3.12/dist-packages/optuna_integration/catboost/catboost.py:103: UserWarning: The reported value is ignored 

...

1100:	learn: 9.4005276	test: 9.2167211	best: 9.2167211 (1100)	total: 11m 11s	remaining: 33.5s


/usr/local/lib/python3.12/dist-packages/optuna_integration/catboost/catboost.py:103: UserWarning: The reported value is ignored because this `step` 1100 is already reported.
  self._trial.report(current_score, step=step)
/usr/local/lib/python3.12/dist-packages/optuna_integration/catboost/catboost.py:103: UserWarning: The reported value is ignored because this `step` 1101 is already reported.
  self._trial.report(current_score, step=step)
/usr/local/lib/python3.12/dist-packages/optuna_integration/catboost/catboost.py:103: UserWarning: The reported value is ignored because this `step` 1102 is already reported.
  self._trial.report(current_score, step=step)
/usr/local/lib/python3.12/dist-packages/optuna_integration/catboost/catboost.py:103: UserWarning: The reported value is ignored because this `step` 1103 is already reported.
  self._trial.report(current_score, step=step)
/usr/local/lib/python3.12/dist-packages/optuna_integration/catboost/catboost.py:103: UserWarning: The reported value

1155:	learn: 9.3970808	test: 9.2142496	best: 9.2142496 (1155)	total: 11m 47s	remaining: 0us

bestTest = 9.214249647
bestIteration = 1155



/usr/local/lib/python3.12/dist-packages/optuna_integration/catboost/catboost.py:103: UserWarning: The reported value is ignored because this `step` 1155 is already reported.
  self._trial.report(current_score, step=step)


0:	learn: 17.9916434	test: 18.0469918	best: 18.0469918 (0)	total: 1.1s	remaining: 21m 16s
100:	learn: 9.5294916	test: 9.3477817	best: 9.3477817 (100)	total: 49.1s	remaining: 8m 32s
200:	learn: 9.4818620	test: 9.3063999	best: 9.3063999 (200)	total: 1m 48s	remaining: 8m 33s
300:	learn: 9.4649839	test: 9.2903442	best: 9.2903442 (300)	total: 2m 45s	remaining: 7m 50s
400:	learn: 9.4534971	test: 9.2799686	best: 9.2799686 (399)	total: 3m 46s	remaining: 7m 5s
500:	learn: 9.4435802	test: 9.2721467	best: 9.2721467 (500)	total: 4m 48s	remaining: 6m 17s
600:	learn: 9.4345547	test: 9.2649675	best: 9.2649672 (599)	total: 5m 52s	remaining: 5m 25s
700:	learn: 9.4266651	test: 9.2592055	best: 9.2592055 (700)	total: 6m 54s	remaining: 4m 29s
800:	learn: 9.4206420	test: 9.2545221	best: 9.2545161 (799)	total: 7m 57s	remaining: 3m 31s
900:	learn: 9.4137141	test: 9.2498295	best: 9.2498295 (900)	total: 9m 2s	remaining: 2m 33s
1000:	learn: 9.4084255	test: 9.2460171	best: 9.2460171 (1000)	total: 10m 7s	remaining